## **TF-IDF & Cosine Similarity**

**TF-IDF: A Measure of Word Importance**

TF-IDF evaluates the importance of a word in a document relative to a collection of documents (corpus). It consists of two components:
- Term Frequency (TF): Measures how often a term appears in a document. Higher frequency indicates greater importance within the document.
- Inverse Document Frequency (IDF): Reduces the weight of terms that appear frequently across many documents, as they are less informative

**Cosine Similarity: Measuring Document Similarity**

Cosine Similarity calculates the cosine of the angle between two vectors in a multidimensional space. It ranges from 0 to 1 for non-negative vectors:
- 1 indicates identical vectors.
- 0 indicates orthogonal vectors (no similarity).

# **Import Libraries**

In [27]:
# import the required libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
# define a function to load the data
def load_data(fname:str) -> pd.DataFrame:
    """
    Loads the data from the given file name and returns a pandas DataFrame.
    
    Parameters:
    fname (str): The name of the file to load the data from.
    
    Returns:
    pd.DataFrame: A DataFrame containing the loaded data.
    """
    # read the data from the file
    df = pd.read_csv(fname)

    # print the original data shape
    print(f'Original data shape: {df.shape}')
    
    return df

In [29]:
# load the movies data
movies_df = load_data('../data/raw/movies.csv')
movies_df.head()

Original data shape: (9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [30]:
# check missing values
print(movies_df.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64


In [31]:
# define the TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_df['genres'])

# similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix)

In [32]:
# mapping title to index
indices = pd.Series(
    movies_df.index,
    index=movies_df['title']
).drop_duplicates()

indices

title
Toy Story (1995)                                0
Jumanji (1995)                                  1
Grumpier Old Men (1995)                         2
Waiting to Exhale (1995)                        3
Father of the Bride Part II (1995)              4
                                             ... 
Black Butler: Book of the Atlantic (2017)    9737
No Game No Life: Zero (2017)                 9738
Flint (2017)                                 9739
Bungo Stray Dogs: Dead Apple (2018)          9740
Andrew Dice Clay: Dice Rules (1991)          9741
Length: 9742, dtype: int64

In [33]:
# define a function to get the recommendations
def recommend_movies(total_movies: int, movie_data: pd.DataFrame, title:str, cosine_sim=cosine_sim) -> list:

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1], # sort by similarity score (descending)
        reverse=True
    )

    # get top k similar movies
    sim_scores = sim_scores[1:total_movies+1]

    # take the indices of the top k similar movies
    movie_indices = [i[0] for i in sim_scores]

    # return the title based on the indices
    print(f'Top {total_movies} similar movies to "{title}":')
    return movie_data['title'].iloc[movie_indices]

In [34]:
# execute the recommendation function
recommend_movies(total_movies=10, movie_data=movies_df, title='Toy Story (1995)')

Top 10 similar movies to "Toy Story (1995)":


1706                                          Antz (1998)
2355                                   Toy Story 2 (1999)
2809       Adventures of Rocky and Bullwinkle, The (2000)
3000                     Emperor's New Groove, The (2000)
3568                                Monsters, Inc. (2001)
6194                                     Wild, The (2006)
6486                               Shrek the Third (2007)
6948                       Tale of Despereaux, The (2008)
7760    Asterix and the Vikings (Astérix et les Viking...
8219                                         Turbo (2013)
Name: title, dtype: str